# `mutate` — Reference
`mutate` creates or overwrites columns using clean keyword argument syntax (`col="expr"` or `col=callable`). Each entry is evaluated in order via pandas `eval()` — a plain formula per column, or a lambda when needed.
### Calling Styles
1. **Keyword arguments** (most Pythonic): `.pt.mutate(bmi="body_mass_g / bill_length_mm ** 2", mass_kg="body_mass_g / 1000")`
2. **Dictionary unpacking** (for new column names with spaces): `.pt.mutate(**{"body mass kg": "body_mass_g / 1000"})`
3. **External file specification**: `.pt.mutate("@features.txt")`
---


In [1]:
import sys, os
_src = os.path.abspath(os.path.join(os.getcwd(), '..', '..', 'src'))
if not os.path.exists(_src): _src = os.path.abspath(os.path.join(os.getcwd(), '..', 'src'))
if _src not in sys.path: sys.path.insert(0, _src)

import numpy as np
import pytae as pt

penguins = pt.sample_data['penguins']

## A single derived column

In [2]:
# Body mass index style ratio — clean assignment formula via kwargs
(
    penguins
    .pt.mutate(bmi='body_mass_g / bill_length_mm ** 2')
    .pt.select('species', 'body_mass_g', 'bill_length_mm', 'bmi')
    .sample(10)
)


,species,body_mass_g,bill_length_mm,bmi
326,Gentoo,4700.0,41.7,2.702874
116,Adelie,2900.0,38.6,1.946361
233,Gentoo,5850.0,48.4,2.497268
270,Gentoo,4850.0,46.6,2.233417
106,Adelie,3750.0,38.6,2.516846
249,Gentoo,5550.0,50.0,2.220000
137,Adelie,3975.0,40.2,2.459716
172,Chinstrap,3600.0,42.4,2.002492
28,Adelie,3150.0,37.9,2.192967
303,Gentoo,5350.0,50.0,2.140000


In [3]:
# Column names inside the expression must stay unquoted — quoting one turns it into
# a string literal, which will break arithmetic
try:
    penguins.pt.mutate(bmi="'body_mass_g' / 'bill_length_mm' ** 2")
except TypeError as exc:
    print(f"TypeError (as expected): {exc}")


TypeError (as expected): unsupported operand type(s) for ** or pow(): 'str' and 'int'


## Multiple entries in one call, and chaining a later entry off an earlier one
Entries are applied left to right, so a later expression can reference a column derived earlier in the *same* `mutate()` call.

In [4]:
# Two independent derived columns in one call via kwargs
(
    penguins
    .pt.mutate(heavy='body_mass_g > 4000', mass_kg='body_mass_g / 1000')
    .pt.select('species', 'body_mass_g', 'mass_kg', 'heavy')
    .sample(10)
)


,species,body_mass_g,mass_kg,heavy
294,Gentoo,4700.0,4.700,True
51,Adelie,4300.0,4.300,True
39,Adelie,4650.0,4.650,True
307,Gentoo,5300.0,5.300,True
179,Chinstrap,3800.0,3.800,False
108,Adelie,3175.0,3.175,False
264,Gentoo,5550.0,5.550,True
158,Chinstrap,3250.0,3.250,False
58,Adelie,2850.0,2.850,False
46,Adelie,3425.0,3.425,False


In [5]:
# mass_lb references mass_kg, derived by the keyword just before it
(
    penguins
    .pt.mutate(mass_kg='body_mass_g / 1000', mass_lb='mass_kg * 2.20462')
    .pt.select('species', 'body_mass_g', 'mass_kg', 'mass_lb')
    .sample(10)
)


,species,body_mass_g,mass_kg,mass_lb
295,Gentoo,5800.0,5.80,12.786796
209,Chinstrap,4050.0,4.05,8.928711
62,Adelie,3600.0,3.60,7.936632
0,Adelie,3750.0,3.75,8.267325
190,Chinstrap,2700.0,2.70,5.952474
171,Chinstrap,4400.0,4.40,9.700328
292,Gentoo,5100.0,5.10,11.243562
289,Gentoo,5550.0,5.55,12.235641
325,Gentoo,5500.0,5.50,12.125410
187,Chinstrap,3900.0,3.90,8.598018


## String comparisons and local variables
String literals *inside* the expression (e.g. `'Adelie'`) need real quotes — column names stay bare. A variable from the calling scope can be referenced with an `@` prefix, same as pandas' own `eval()`/`query()`.


In [6]:
# String comparisons inside the expression formula
(
    penguins
    .pt.mutate(is_adelie="species == 'Adelie'")
    .pt.select('species', 'is_adelie')
    .sample(10)
)


,species,is_adelie
142,Adelie,True
41,Adelie,True
62,Adelie,True
97,Adelie,True
247,Gentoo,False
129,Adelie,True
199,Chinstrap,False
220,Gentoo,False
327,Gentoo,False
147,Adelie,True


In [7]:
# @-prefixed names resolve against the scope that called mutate(), not the module
# — works identically whether calling pt.mutate() or chaining .pt.mutate()
threshold = 4000

(
    penguins
    .pt.mutate(heavy='body_mass_g >= @threshold')
    .pt.select('species', 'body_mass_g', 'heavy')
    .sample(10)
)


,species,body_mass_g,heavy
341,Gentoo,5750.0,True
27,Adelie,3200.0,False
175,Chinstrap,3800.0,False
54,Adelie,2900.0,False
312,Gentoo,4750.0,True
307,Gentoo,5300.0,True
120,Adelie,3150.0,False
339,Gentoo,NaN,False
156,Chinstrap,3725.0,False
47,Adelie,2975.0,False


## Overwriting an existing column

In [8]:
# mutate() can overwrite a column in place, e.g. converting units
(
    penguins
    .pt.mutate(body_mass_g='body_mass_g / 1000')
    .pt.select('species', 'body_mass_g')
    .sample(10)
)


,species,body_mass_g
183,Chinstrap,4.30
120,Adelie,3.15
229,Gentoo,5.15
26,Adelie,3.55
332,Gentoo,4.65
333,Gentoo,5.50
124,Adelie,3.05
145,Adelie,3.65
173,Chinstrap,3.40
330,Gentoo,5.00


## Column names with spaces — bracket syntax
Use square brackets `[col name]` to reference column names containing spaces without shell or syntax hazards (backticks are also supported):

In [9]:
import pandas as pd
spaced = pd.DataFrame({'body mass g': [3750, 4200], 'bill length mm': [39.1, 46.5]})

# Square brackets protect column names that contain spaces
spaced.pt.mutate(bmi="[body mass g] / [bill length mm] ** 2")


,body mass g,bill length mm,bmi
0,3750,39.1,2.452888
1,4200,46.5,1.942421


## Real-world pipeline: mutate → filter → select
`mutate()` chains like any other pytae method — filter on a column you just derived with `qry()`.

In [10]:
(
    penguins
    .pt.mutate(bmi='body_mass_g / bill_length_mm ** 2')
    .pt.qry(bmi='> 2')
    .pt.select('species', 'island', 'body_mass_g', 'bill_length_mm', 'bmi')
    .sample(10)
)


,species,island,body_mass_g,bill_length_mm,bmi
33,Adelie,Dream,3900.0,40.9,2.331406
247,Gentoo,Biscoe,5650.0,47.8,2.472821
60,Adelie,Biscoe,3150.0,35.7,2.471577
306,Gentoo,Biscoe,4600.0,43.4,2.442184
272,Gentoo,Biscoe,4400.0,45.1,2.163215
230,Gentoo,Biscoe,4650.0,40.9,2.779754
281,Gentoo,Biscoe,5300.0,46.2,2.483087
35,Adelie,Dream,4150.0,39.2,2.700698
127,Adelie,Torgersen,4300.0,41.5,2.496734
41,Adelie,Dream,3900.0,40.8,2.342849


## Conditional column creation — `if_else()`, `case_when()`, `map()`
Plain `eval()` has no ternary/`where()` support, so `mutate()` provides three dplyr-style helpers as ordinary function calls, evaluated via `np.where()`/`np.select()`/`Series.map()`: `if_else(condition, true_value, false_value)`, `case_when((cond1, val1), (cond2, val2), ..., default)`, and `map(column, {key: value, ...}, default)`. A trailing bare argument to `case_when` is the catch-all default (like SQL ELSE). Because they are ordinary calls, they compose and chain with each other and with any pandas method. String outcomes need quotes; conditions are vectorized — prefer `and`/`or`/`not` (the bitwise `&`/`|`/`~` also work).

In [11]:
# if_else(condition, true_value, false_value) — like dplyr's if_else()
# true_value and false_value can be scalars or column names; conditions are vectorized
(
    penguins
    .pt.mutate(weight_class="if_else(body_mass_g > 4000, 'heavy', 'light')")
    .pt.select('species', 'body_mass_g', 'weight_class')
    .sample(10)
)


,species,body_mass_g,weight_class
101,Adelie,4725.0,heavy
46,Adelie,3425.0,light
77,Adelie,3900.0,light
131,Adelie,3500.0,light
92,Adelie,3400.0,light
52,Adelie,3450.0,light
105,Adelie,3550.0,light
79,Adelie,4000.0,light
126,Adelie,3275.0,light
42,Adelie,3100.0,light


In [12]:
# case_when supports tuples or flat pairs with default=
(
    penguins
    .pt.mutate(
        size_class="case_when(body_mass_g >= 4500, 'large', body_mass_g >= 3500, 'medium', default='small')"
    )
    .pt.select('species', 'body_mass_g', 'size_class')
    .sample(10)
)


,species,body_mass_g,size_class
301,Gentoo,5450.0,large
322,Gentoo,4975.0,large
235,Gentoo,5850.0,large
17,Adelie,4500.0,large
200,Chinstrap,3250.0,small
228,Gentoo,4400.0,medium
226,Gentoo,4800.0,large
258,Gentoo,4350.0,medium
39,Adelie,4650.0,large
178,Chinstrap,3400.0,small


In [13]:
# map(column, {key: value, ...}, default) — recode a column through a dictionary lookup
# unmapped keys become default if given, else NaN
(
    penguins
    .pt.mutate(
        island_code="map(island, {'Torgersen': 'TOR', 'Biscoe': 'BIS'}, 'OTH')"
    )
    .pt.select('species', 'island', 'island_code')
    .sample(10)
)


,species,island,island_code
71,Adelie,Torgersen,TOR
262,Gentoo,Biscoe,BIS
299,Gentoo,Biscoe,BIS
265,Gentoo,Biscoe,BIS
165,Chinstrap,Dream,OTH
290,Gentoo,Biscoe,BIS
20,Adelie,Biscoe,BIS
181,Chinstrap,Dream,OTH
210,Chinstrap,Dream,OTH
45,Adelie,Dream,OTH


## First non-null resolution — `coalesce()`

`coalesce(col1, col2, ..., default)` evaluates candidates left-to-right and returns the first non-null value per row, like SQL `COALESCE()` or `dplyr::coalesce()`:

In [14]:
contacts = pd.DataFrame({
    'mobile': [None, '555-1234', None],
    'home': ['555-5678', None, None],
    'work': [None, None, '555-9012'],
})

contacts.pt.mutate(preferred_contact="coalesce(mobile, home, work, 'N/A')")


,mobile,home,work,preferred_contact
0,NaN,555-5678,NaN,555-5678
1,555-1234,NaN,NaN,555-1234
2,NaN,NaN,555-9012,555-9012


## Combining expressions and callables in kwargs

`mutate()` cleanly mixes expression strings and python callables (e.g. lambdas) in keyword arguments:


In [15]:
(
    penguins
    .pt.mutate(
        bmi='body_mass_g / bill_length_mm ** 2',
        mass_kg='body_mass_g / 1000',
        is_heavy=lambda df: df['body_mass_g'] > 4000,
    )
    .pt.select('species', 'bmi', 'mass_kg', 'is_heavy')
    .head(5)
)


,species,bmi,mass_kg,is_heavy
0,Adelie,2.452888,3.75,False
1,Adelie,2.435507,3.80,False
2,Adelie,2.001121,3.25,False
3,Adelie,NaN,NaN,False
4,Adelie,2.561456,3.45,False


### Row-wise custom logic with `lambda df: df.apply(func, axis=1)`

When custom logic requires multi-column `if/elif/else` branching or arbitrary Python functions expecting a single row (a `pandas.Series`), pass `lambda df: df.apply(custom_func, axis=1)`.

Because `mutate()` executes sequentially, the row function can even access columns created earlier in the very same `mutate()` call (such as `row["mass_kg"]`):

In [16]:
def classify_penguin(row):
    if pd.isna(row["mass_kg"]):
        return "Incomplete Data"
    if row["species"] == "Gentoo" and row["mass_kg"] > 5.0:
        return "Giant Gentoo"
    elif row["island"] == "Torgersen" and row["mass_kg"] > 3.7:
        return "Prime Torgersen"
    else:
        return "Standard"

(
    penguins
    .pt.mutate(
        mass_kg="body_mass_g / 1000",
        category=lambda df: df.apply(classify_penguin, axis=1),
    )
    .pt.select("species", "island", "body_mass_g", "mass_kg", "category")
    .head(6)
)


,species,island,body_mass_g,mass_kg,category
0,Adelie,Torgersen,3750.0,3.75,Prime Torgersen
1,Adelie,Torgersen,3800.0,3.80,Prime Torgersen
2,Adelie,Torgersen,3250.0,3.25,Standard
3,Adelie,Torgersen,NaN,NaN,Incomplete Data
4,Adelie,Torgersen,3450.0,3.45,Standard
5,Adelie,Torgersen,3650.0,3.65,Standard


## Loading specs from an external file — `@specs.txt`
For complex feature engineering or shared pipelines across Python and the CLI, specs can be loaded from an external file. The file supports multiline expressions, `#` comments, empty lines, and column chaining (where later formulas reference columns defined earlier in the same file).
### File content of `penguin_features.txt`:
```text
# Engineering features for penguins
mass_kg = body_mass_g / 1000
# Convert kg to lbs (references mass_kg computed above)
mass_lb = mass_kg * 2.20462
```


In [17]:
# Write a small demo spec file with comments and assignment '=' syntax
spec_content = """# Engineering features for penguins
mass_kg = body_mass_g / 1000

# Convert kg to lbs
mass_lb = mass_kg * 2.20462
"""
with open("penguin_features.txt", "w") as f:
    f.write(spec_content)

# Load directly using @filename
result = penguins.pt.mutate("@penguin_features.txt")
out = (
    result
    .pt.select("species", "mass_kg", "mass_lb")
    .sample(5)
)

# Clean up demo file
import os
os.remove("penguin_features.txt")

out


,species,mass_kg,mass_lb
260,Gentoo,3.950,8.708249
75,Adelie,4.250,9.369635
1,Adelie,3.800,8.377556
223,Gentoo,5.700,12.566334
123,Adelie,3.875,8.542902


## Columns with spaces — SQL-style brackets `[col]`

SQL-style square brackets `[col a]` are supported alongside backticks, avoiding shell backtick substitution hazards:

In [18]:
spaced.pt.mutate(ratio='[body mass g] / [bill length mm]')


,body mass g,bill length mm,ratio
0,3750,39.1,95.907928
1,4200,46.5,90.322581


## Creating new columns with spaces — string expressions `[col] = ...`

You can create columns with spaces directly using string assignment expressions with brackets `[new col] = ...` with zero dictionary unpacking.

(Dictionary unpacking `**{'col name': ...}` is also supported for kwargs).

In [19]:
# Create a new column with spaces using bracket string expression (no dict unpacking!)
(
    spaced
    .pt.mutate("[body mass ratio] = [body mass g] / [bill length mm]")
    .pt.select('[body mass g]', '[body mass ratio]')
    .head()
)

,body mass g,body mass ratio
0,3750,95.907928
1,4200,90.322581


In [20]:
# Define multiple new columns with spaces at once via string expressions
(
    penguins
    .pt.mutate(
        "[body mass kg] = body_mass_g / 1000",
        "[bill ratio] = bill_length_mm / bill_depth_mm",
    )
    .pt.select('species', '[body mass kg]', '[bill ratio]')
    .head()
)

,species,body mass kg,bill ratio
0,Adelie,3.75,2.090909
1,Adelie,3.80,2.270115
2,Adelie,3.25,2.238889
3,Adelie,NaN,NaN
4,Adelie,3.45,1.901554


Note: When loading specs from an external file (`@specs.txt`) or via the CLI (`-mutate`), names with spaces can be written directly on the left-hand side of `=` without dictionary unpacking:

```text
# inside an external spec file or CLI flag:
my bmi = [body mass g] / [bill length mm] ** 2
```


## Grouped mutations — `mutate(..., by=...)`

Unlike `pt.agg()` which collapses rows ($N \to K$), `mutate(..., by=...)` evaluates window calculations and group-level summaries and broadcasts them back to every row ($N \to N$).

Supported features:
- **Injected aggregation functions**: `mean(x)`, `sum(x)`, `median(x)`, `min(x)`, `max(x)`, `std(x)`, `var(x)`
- **Group row count**: `n` or `n()` (dplyr / Polars convention)
- **Sequential expressions**: Later expressions within the same `mutate` call can immediately reference earlier group-derived columns (e.g. `diff = tip - avg_tip`)
- **Arbitrary output column names**: Name target columns cleanly without hardcoded prefixes
- **Multi-column grouping**: `by=['species', 'island']` or `by='species, island'`
- **Spaced column names**: `[total bill]`
- **Missing group handling**: `dropna=False` retains NA groups


In [21]:
# Compute group average tip, deviation from group average, and group sample size
tips = pt.sample("tips")
(
    tips
    .pt.select("day", "time", "total_bill", "tip")
    .pt.mutate(
        "avg_tip = mean(tip), diff = tip - avg_tip, group_size = n",
        by="day",
    )
    .head(6)
)

,day,time,total_bill,tip,avg_tip,diff,group_size
0,Sun,Dinner,16.99,1.01,3.255132,-2.245132,76
1,Sun,Dinner,10.34,1.66,3.255132,-1.595132,76
2,Sun,Dinner,21.01,3.50,3.255132,0.244868,76
3,Sun,Dinner,23.68,3.31,3.255132,0.054868,76
4,Sun,Dinner,24.59,3.61,3.255132,0.354868,76
5,Sun,Dinner,25.29,4.71,3.255132,1.454868,76


In [22]:
# Multi-column grouping with spaced columns
(
    tips
    .rename(columns={"total_bill": "total bill"})
    .pt.select("day", "time", "total bill", "tip")
    .pt.mutate(
        "[avg bill] = mean([total bill]), [bill diff] = [total bill] - [avg bill], count = n()",
        by=["day", "time"],
    )
    .head(6)
)

,day,time,total bill,tip,avg bill,bill diff,count
0,Sun,Dinner,16.99,1.01,21.41,-4.42,76
1,Sun,Dinner,10.34,1.66,21.41,-11.07,76
2,Sun,Dinner,21.01,3.50,21.41,-0.40,76
3,Sun,Dinner,23.68,3.31,21.41,2.27,76
4,Sun,Dinner,24.59,3.61,21.41,3.18,76
5,Sun,Dinner,25.29,4.71,21.41,3.88,76


In [23]:
# De-meaning and z-score calculation per group
(
    penguins
    .pt.select("species", "body_mass_g")
    .pt.mutate(
        "mean_mass = mean(body_mass_g), std_mass = std(body_mass_g), z_score = (body_mass_g - mean_mass) / std_mass",
        by="species",
    )
    .head(6)
)

,species,body_mass_g,mean_mass,std_mass,z_score
0,Adelie,3750.0,3700.662252,458.566126,0.107591
1,Adelie,3800.0,3700.662252,458.566126,0.216627
2,Adelie,3250.0,3700.662252,458.566126,-0.982764
3,Adelie,NaN,3700.662252,458.566126,NaN
4,Adelie,3450.0,3700.662252,458.566126,-0.546622
5,Adelie,3650.0,3700.662252,458.566126,-0.110480


### Handling missing grouping values with `dropna`

By default, `dropna=False` computes group statistics across `NA` rows as their own distinct group (no silent data loss). Pass `dropna=True` to exclude rows with `NA` in grouping columns from calculations, filling `NaN` in mutated columns while preserving total row count and index alignment:


In [24]:
sample_data = pd.DataFrame({
    "grp": ["A", "A", None, "B", None],
    "val": [10.0, 20.0, 100.0, 30.0, 200.0]
})

# dropna=False (default): NA rows form a distinct group
print("dropna=False (default):")
print(sample_data.pt.mutate("avg = mean(val), sz = n", by="grp"))

# dropna=True: NA groups receive NaN
print("\ndropna=True:")
print(sample_data.pt.mutate("avg = mean(val), sz = n", by="grp", dropna=True))


dropna=False (default):
   grp    val    avg  sz
0    A   10.0   15.0   2
1    A   20.0   15.0   2
2  NaN  100.0  150.0   2
3    B   30.0   30.0   1
4  NaN  200.0  150.0   2

dropna=True:
   grp    val   avg   sz
0    A   10.0  15.0  2.0
1    A   20.0  15.0  2.0
2  NaN  100.0   NaN  NaN
3    B   30.0  30.0  1.0
4  NaN  200.0   NaN  NaN


## Comprehensive end-to-end recipe: combining all `mutate` capabilities

This master example combines all core features in a single fluent pipeline:
1. **Normal mutate**: Standard arithmetic derivation (`bill_ratio`, `mass_kg`).
2. **Sequential column referencing**: Using a freshly created column (`mass_kg`) inside subsequent expressions (`bmi`, `above_target`, `size_label`) within the *same* `mutate()` call.
3. **Binary conditional (`if_else`)**: Conditional branching into `'Heavy'` / `'Standard'`.
4. **Multi-condition tiers (`case_when`)**: Multi-tier categorization with ordered tuples and a fallback `default`.
5. **Null fallback (`coalesce`)**: Resolving the first non-null value across columns or supplying a fallback for missing data.
6. **Dictionary mapping (`map`)**: Category recoding through a dictionary lookup with fallback handling.
7. **Local variables (`@var`)**: Referencing caller-scope Python variables directly using the `@` prefix.
8. **Columns with spaces (`[col]`)**: Defining spaced columns via dict unpacking and referencing them with brackets `[mass lbs]`.
9. **Grouped window transforms (`by=`)**: Computing group-level aggregations (`mean(mass_kg)`), relative offsets (`mass_diff`), and group sizes (`n`) without collapsing rows.
10. **Lambda callables**: Custom function inspecting the accumulated DataFrame with all prior mutations already attached.


In [25]:
import numpy as np
import pytae as pt

penguins = pt.sample("penguins")

# 7. Local Python variable from caller scope (referenced with @ prefix)
target_threshold = 4.2

master_df = (
    penguins
    .pt.mutate(
        # 1. Normal mutate: standard arithmetic derivation
        bill_ratio="bill_length_mm / bill_depth_mm",
        mass_kg="body_mass_g / 1000",
        
        # 3. Referencing newly created col (mass_kg) in the same mutate call
        bmi="mass_kg / ((flipper_length_mm / 1000) ** 2)",
        
        # 7a. Local variable reference (@target_threshold) from caller scope
        above_target="mass_kg > @target_threshold",
        
        # 4. if_else: binary conditional logic
        size_label="if_else(mass_kg >= 4.5, 'Heavy', 'Standard')",
        
        # 5. case_when: multi-condition tiered categorization
        weight_tier="case_when((mass_kg >= 5.0, 'XL'), (mass_kg >= 4.0, 'Large'), (mass_kg >= 3.0, 'Medium'), default='Small')",
        
        # 6. coalesce: prioritize sex, fall back to 'Undetermined' when null (row 3 has sex=NaN)
        effective_sex="coalesce(sex, 'Undetermined')",
        
        # 7b. map(): category dictionary lookup with default fallback
        species_code="map(species, {'Adelie': 'AD', 'Gentoo': 'GE', 'Chinstrap': 'CH'}, 'OTHER')",
        
        # 7c. Spaced column creation via dict unpacking & bracket referencing
        **{"mass lbs": "mass_kg * 2.20462"},
        heavy_lbs="[mass lbs] > 9.5",
        
        # 9. Grouped window transforms via by= without collapsing rows
        species_avg_mass="mean(mass_kg)",
        mass_diff="mass_kg - species_avg_mass",
        group_size="n",
        
        # 2. lambda: custom callable inspecting the accumulated DataFrame
        is_large=lambda df: (df["mass_kg"] > 4.0) & (df["bill_ratio"] > 2.5),
        
        by="species",
    )
)

cols_to_show = [
    "species", "species_code", "mass_kg", "species_avg_mass", "mass_diff",
    "group_size", "bmi", "size_label", "weight_tier", "effective_sex", 
    "above_target", "heavy_lbs", "is_large"
]
master_df[cols_to_show].head(8)


,species,species_code,mass_kg,species_avg_mass,mass_diff,group_size,bmi,size_label,weight_tier,effective_sex,above_target,heavy_lbs,is_large
0,Adelie,AD,3.750,3.700662,0.049338,152,114.465370,Standard,Medium,Male,False,False,False
1,Adelie,AD,3.800,3.700662,0.099338,152,109.839288,Standard,Medium,Female,False,False,False
2,Adelie,AD,3.250,3.700662,-0.450662,152,85.470085,Standard,Medium,Female,False,False,False
3,Adelie,AD,NaN,3.700662,NaN,152,NaN,Standard,Small,Undetermined,False,False,False
4,Adelie,AD,3.450,3.700662,-0.250662,152,92.619936,Standard,Medium,Female,False,False,False
5,Adelie,AD,3.650,3.700662,-0.050662,152,101.108033,Standard,Medium,Male,False,False,False
6,Adelie,AD,3.625,3.700662,-0.075662,152,110.649858,Standard,Medium,Female,False,False,False
7,Adelie,AD,4.675,3.700662,0.974338,152,122.945431,Heavy,Large,Male,True,True,False
